# Baseline: vanilla TRELLIS.2 (no restyle)

Runs the plain `microsoft/TRELLIS.2-4B` image-to-3D pipeline directly on your input images, with **no restyle preprocessing** — this is the "theirs" baseline to compare against your restyle pipeline.

**Runtime → Change runtime type → GPU (A100 / L4).**

Flow: setup TRELLIS.2 → load pipeline → run on each input image → download the baseline `.glb`s. You then score them locally with `eval/score_baseline.py` (same CLIP / Gen3DEval / ULIP metrics as the rest of the eval).

Inputs needed: a zip of your input images (e.g. `eval/dataset/images/`) named exactly as `captions.csv` filenames (`bicycle_01.png`, ...).

In [ ]:
# 0. Confirm GPU
!nvidia-smi -L

In [ ]:
# 1. Clone TRELLIS.2 and install (all CUDA extensions needed for inference).
#    ~15-20 min the first time. Do NOT pass --new-env in Colab.
%cd /content
!git clone -b main https://github.com/microsoft/TRELLIS.2.git --recursive
%cd /content/TRELLIS.2
import os
os.environ['TORCH_CUDA_ARCH_LIST'] = '8.0;8.6;8.9;9.0'
os.environ['MAX_JOBS'] = '4'
os.environ['OPENCV_IO_ENABLE_OPENEXR'] = '1'
!. ./setup.sh --basic --flash-attn --o-voxel --cumesh --flexgemm --nvdiffrast --nvdiffrec

In [ ]:
# 2. Load the pipeline (downloads the TRELLIS.2-4B weights on first run).
%cd /content/TRELLIS.2
import os
os.environ['OPENCV_IO_ENABLE_OPENEXR'] = '1'
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
from trellis2.pipelines import Trellis2ImageTo3DPipeline
import o_voxel

pipeline = Trellis2ImageTo3DPipeline.from_pretrained('microsoft/TRELLIS.2-4B')
pipeline.cuda()
print('pipeline ready')

In [ ]:
# 3. Upload a zip of your input images (eval/dataset/images/*.png).
#    Locally:  cd eval/dataset && zip -r images.zip images
from google.colab import files
import zipfile, os
os.makedirs('/content/inputs', exist_ok=True)
up = files.upload()  # pick images.zip
for name in up:
    if name.endswith('.zip'):
        with zipfile.ZipFile(name) as z:
            z.extractall('/content/inputs')
# flatten: find all image files
import glob
imgs = sorted(glob.glob('/content/inputs/**/*.png', recursive=True) +
              glob.glob('/content/inputs/**/*.jpg', recursive=True))
print(f'{len(imgs)} input images')
print('\n'.join(os.path.basename(p) for p in imgs))

In [ ]:
# 4. Run vanilla TRELLIS.2 on each input (NO restyle). Saves baseline_glbs/<stem>.glb.
#    Resumable: skips inputs whose .glb already exists.
from PIL import Image
import os, traceback
OUT = '/content/baseline_glbs'
os.makedirs(OUT, exist_ok=True)

for i, p in enumerate(imgs, 1):
    stem = os.path.splitext(os.path.basename(p))[0]
    out_glb = f'{OUT}/{stem}.glb'
    if os.path.exists(out_glb):
        print(f'[{i}/{len(imgs)}] skip (done) {stem}'); continue
    print(f'[{i}/{len(imgs)}] {stem} ...')
    try:
        image = Image.open(p).convert('RGB')
        mesh = pipeline.run(image)[0]
        mesh.simplify(16777216)  # nvdiffrast limit
        glb = o_voxel.postprocess.to_glb(
            vertices=mesh.vertices, faces=mesh.faces, attr_volume=mesh.attrs,
            coords=mesh.coords, attr_layout=mesh.layout, voxel_size=mesh.voxel_size,
            aabb=[[-0.5,-0.5,-0.5],[0.5,0.5,0.5]],
            decimation_target=1000000, texture_size=4096,
            remesh=True, remesh_band=1, remesh_project=0, verbose=False)
        glb.export(out_glb, extension_webp=True)
        print(f'    -> {out_glb}')
    except Exception:
        traceback.print_exc()

In [ ]:
# 5. Zip the baseline meshes and download. Then score them locally:
#    unzip into eval/baseline_glbs/ and run:
#      .venv/bin/python eval/score_baseline.py eval/baseline_glbs eval/results_baseline.csv
import shutil
shutil.make_archive('/content/baseline_glbs', 'zip', '/content/baseline_glbs')
from google.colab import files
files.download('/content/baseline_glbs.zip')